# Differentiable Vanderlick output layer (proof of concept)

Instead of the network outputting a dyad probability directly, it outputs a
per-position **binding energy** `E(x)`, which is passed through the **Vanderlick
recursion implemented in PyTorch** as the final layer. The recursion is
differentiable, so gradients flow back through the physics, and it enforces
**steric exclusion exactly**: the network can never place two nucleosomes within
one footprint. The network learns the sequence/methylation -> energy map; the
physics layer turns that into an occupancy that respects the hard-rod mechanics.

This is a proof of concept: the Vanderlick recursion is an O(L) sequential scan,
so it is slow and we train on short fibers.

In [ ]:
import sys, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

ROOT = Path.cwd()
if not (ROOT / "nuctool").is_dir() and (ROOT.parent / "nuctool").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "nuctool"))
from ChromatinFibers import SimulationParams, simulate_chromatin_fibers, read_simulation_results, compute_vanderlick

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FOOTPRINT = 146
WORK = ROOT / "data" / "vanderlick_poc"; WORK.mkdir(parents=True, exist_ok=True)
print("device:", device)

In [ ]:
# ── Differentiable Vanderlick layer ─────────────────────────────────────────
# forward[i] = exp(E[i] - windowed_sum_{i-fp..i-1} forward);  backward mirrors it.
# dyads[i] = forward[i] * backward_reversed[i]  (the equilibrium dyad probability).
def vanderlick_layer(E, footprint=FOOTPRINT):
    """E: (B, L) per-position energy (already sign-oriented so higher E = more favourable).
    Returns dyads (B, L) in [0,1] and occupancy (B, L)."""
    B, L = E.shape
    fwd = []; S = torch.zeros(B, dtype=E.dtype, device=E.device)
    for i in range(L):
        f = torch.exp(E[:, i] - S); fwd.append(f)
        S = S + f
        if i >= footprint: S = S - fwd[i - footprint]
    forward = torch.stack(fwd, dim=1)
    rf = forward.flip(1); bwd = []; Sp = torch.zeros(B, dtype=E.dtype, device=E.device)
    for i in range(L):
        b = 1.0 - Sp; bwd.append(b)
        Sp = Sp + rf[:, i] * b
        if i >= footprint: Sp = Sp - rf[:, i - footprint] * bwd[i - footprint]
    backward = torch.stack(bwd, dim=1)
    dyads = torch.clamp(forward * backward.flip(1), min=0.0)
    ker = torch.ones(1, 1, footprint, dtype=E.dtype, device=E.device)
    occ = F.conv1d(dyads.unsqueeze(1), ker, padding=footprint // 2).squeeze(1)[:, :L]
    return dyads, torch.clamp(occ, 0, 1)

# self-check against the numpy reference (compute_vanderlick uses free_energy = -wrapping_energy)
_we = np.random.RandomState(0).randn(400) * 3.0
_dy_np, _ = compute_vanderlick(_we.copy(), show=False)
_dy_t, _ = vanderlick_layer(torch.tensor(-_we, dtype=torch.float64).unsqueeze(0), FOOTPRINT)
print("torch vs numpy dyads max abs diff:", float(np.max(np.abs(_dy_t.numpy()[0] - _dy_np))))

In [ ]:
# ── Network: sequence/methylation -> energy -> Vanderlick -> dyad probability ─
class VanderlickDyadNet(nn.Module):
    def __init__(self, vocab=8, emb=32, hid=64, footprint=FOOTPRINT, energy_scale=5.0):
        super().__init__()
        self.embedding = nn.Embedding(vocab, emb)
        self.conv1 = nn.Conv1d(emb, hid, 7, padding=3,  dilation=1); self.bn1 = nn.BatchNorm1d(hid)
        self.conv2 = nn.Conv1d(hid, hid, 7, padding=6,  dilation=2); self.bn2 = nn.BatchNorm1d(hid)
        self.conv3 = nn.Conv1d(hid, hid, 7, padding=12, dilation=4); self.bn3 = nn.BatchNorm1d(hid)
        self.head  = nn.Conv1d(hid, 1, 1)
        self.footprint = footprint; self.energy_scale = energy_scale
    def forward(self, x):
        h = self.embedding(x).permute(0, 2, 1)
        h = torch.relu(self.bn1(self.conv1(h)))
        h = torch.relu(self.bn2(self.conv2(h)))
        h = torch.relu(self.bn3(self.conv3(h)))
        E = self.energy_scale * torch.tanh(self.head(h).squeeze(1))   # (B, L) bounded energy
        dyads, occ = vanderlick_layer(E, self.footprint)
        return dyads.clamp(0.0, 1.0), occ, E   # clamp to a valid probability

In [ ]:
# ── short simulated fibers for the POC (recursion is O(L), so keep L small) ──
LENGTH   = 600
N_SAMPLES = 400
sim_path = WORK / f"poc_L{LENGTH}_n{N_SAMPLES}.h5"
if not sim_path.exists():
    simulate_chromatin_fibers(SimulationParams(
        n_samples=N_SAMPLES, length_bp=LENGTH, amplitude=0.05, chemical_potential_kT=6,
        padding_bp=500, motifs=["A", "T"], strand="both", efficiency=0.2), str(sim_path))
print("simulated:", sim_path, "|", read_simulation_results(str(sim_path)).n_samples, "fibers")

class DS(Dataset):
    def __init__(self, path, idx):
        self.path=str(path); self.idx=idx; self.L=read_simulation_results(self.path).length_bp
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        d, e, _ = read_simulation_results(self.path, self.idx[i])
        e = np.asarray(e, np.int64); lab = np.zeros(len(e), np.float32)
        for pos in np.asarray(d, int):
            if 0 <= pos < len(e): lab[pos] = 1.0
        s = torch.LongTensor(e); l = torch.FloatTensor(lab)
        if len(e) < self.L:
            s = F.pad(s, (0, self.L - len(e)), value=0); l = F.pad(l, (0, self.L - len(e)), value=-1)
        return s[:self.L], l[:self.L]

n = read_simulation_results(str(sim_path)).n_samples
tr_idx = list(range(int(0.85 * n))); te_idx = list(range(int(0.85 * n), n))
train_dl = DataLoader(DS(sim_path, tr_idx), batch_size=4, shuffle=True)

In [ ]:
# ── train end-to-end: weighted BCE on the physics-produced dyad probability ──
model = VanderlickDyadNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
EPOCHS, MAX_BATCHES, POS_W = 5, 40, 200.0

for ep in range(EPOCHS):
    model.train(); tot = nb = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        dyads, occ, E = model(xb)
        p = dyads.clamp(1e-6, 1 - 1e-6)
        y = yb.clamp(0, 1); mask = (yb >= 0).float()
        w = torch.where(yb > 0.5, torch.tensor(POS_W, device=device), torch.tensor(1.0, device=device))
        loss = -(w * (y * torch.log(p) + (1 - y) * torch.log(1 - p)) * mask).sum() / mask.sum().clamp(min=1)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot += loss.item(); nb += 1
        if nb >= MAX_BATCHES: break
    print(f"epoch {ep+1}/{EPOCHS}  loss={tot/max(nb,1):.4f}")
print("done")

In [ ]:
# ── evaluate: prediction example + positional-error distribution ────────────
model.eval()
def predict(enc):
    with torch.no_grad():
        dyads, occ, E = model(torch.LongTensor(enc).unsqueeze(0).to(device))
    return dyads.squeeze(0).cpu().numpy(), occ.squeeze(0).cpu().numpy()

# one held-out fiber
d_true, enc, _ = read_simulation_results(str(sim_path), te_idx[0]); d_true = np.array(d_true, int)
p, occ = predict(enc)
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].fill_between(np.arange(len(p)), p, color="orange", alpha=0.6, label="P(dyad) (physics layer)")
ax[0].vlines(d_true, 0, 1.05, color="black", ls="dotted", label="true dyad")
ax[0].set_ylabel("P(dyad)"); ax[0].legend(fontsize=8)
ax[1].fill_between(np.arange(len(occ)), occ, color="steelblue", alpha=0.6, label="occupancy (physics layer)")
ax[1].set_ylabel("occupancy"); ax[1].set_xlabel("Position (bp)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# error distribution over held-out fibers
errs = []
for idx in te_idx:
    dt, e, _ = read_simulation_results(str(sim_path), idx); dt = np.array(dt, int)
    p, _ = predict(e)
    called, _ = find_peaks(p, distance=FOOTPRINT, height=max(0.05, p.max()*0.3))
    for c in called:
        if len(dt): errs.append(int(c - dt[np.argmin(np.abs(dt - c))]))
errs = np.array(errs)
plt.figure(figsize=(8, 3))
plt.hist(errs, bins=np.arange(-30, 31, 2), color="steelblue", edgecolor="none")
plt.axvline(0, color="k", lw=0.8)
plt.xlabel("Positional error (bp): called - true"); plt.ylabel("count")
plt.title(f"median |error| = {np.median(np.abs(errs)):.1f} bp (n={len(errs)} calls)" if len(errs) else "no peaks called yet")
plt.tight_layout(); plt.show()